# 16 — Typing, dataclasses and protocols

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/python/16-typing-and-dataclasses/lab.ipynb) [![Open in LinguAgent](https://img.shields.io/badge/Open%20in-LinguAgent-1f6f5c)](https://linguagent.com/courses/python/typing-and-dataclasses?utm_source=github&utm_medium=labs)

*~45 minutes · 31 tests*

Type hints do not change how Python runs — they are erased at import time
and enforced by nothing at runtime. What they change is how much of your
design a reader (or a type checker, or an interviewer skimming your
screen-share) can see without running the code. `@dataclass` and `Protocol`
are the two tools that turn hints into actual leverage: less boilerplate for
the classes that are just data, and structural interfaces that do not need
inheritance to type-check.

This is the module that makes mid-to-senior code reviews go faster —
"why not just use a dataclass here" and "does this need to inherit from
anything" are both answered below.

Work in `exercises.py`.

**How this notebook works:** run the first cell; then, section by section, read, write the code in the cell under it, and run the check below it. Stuck? `lab.solution("name")` shows the reference — try first, it's worth it.

In [ ]:
#@title ▶️ Run me first — fetches this lab and its checker
import os, sys, subprocess
HOME = "/content/linguagent-labs"
if "google.colab" in sys.modules:
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/mrshahabian/linguagent-labs.git", HOME], check=True)
else:  # Jupyter, VS Code or a Codespace: this notebook already sits inside the repo
    HOME = os.path.abspath(os.path.join(os.getcwd(), *[".."] * 3)) if os.path.isdir("../../../labkit") else os.getcwd()
sys.path.insert(0, HOME)
import labkit
lab = labkit.open("python/typing-and-dataclasses")

In [ ]:
# What the exercises below use — run this cell.
from dataclasses import dataclass, field

from typing import Callable, Generic, Iterable, Optional, Protocol, TypeVar, runtime_checkable

T = TypeVar("T")

## What you'll build

| Name | You'll implement | Shows up in real code as |
|---|---|---|
| `safe_divide` | `safe_divide` | Modelling failure as `Optional[T]` instead of an exception the caller must remember to catch |
| `Stack` | `push`, `pop`, `peek`, `is_empty`, `__len__` | A generic container a type checker can specialise per call site |
| `first_match` | `first_match` | A generic helper whose element type flows from input to predicate |
| `Point` | `distance_to` | An immutable, hashable value type — `frozen=True` |
| `SemVer` | `__str__` | Auto-generated ordering, with one field excluded from comparison |
| `sort_versions` | `sort_versions` | Sorting on a dataclass's generated `__lt__` |
| `ShoppingCart` | `add_item`, `total_items` | `default_factory` — the fix for the mutable-default-argument bug |
| `PositiveInt` | `__post_init__` | Validating a dataclass after construction |
| `Circle`, `Rectangle` | `area` | Two unrelated classes that satisfy the same `Protocol` |
| `total_area` | `total_area` | A function typed against a `Protocol`, not a base class |
| `satisfies_supports_area` | `satisfies_supports_area` | `isinstance` against a `@runtime_checkable` `Protocol` |

---

## 1. Type hints are erased, not enforced

```python
def safe_divide(a: float, b: float) -> Optional[float]:
    if b == 0:
        return None
    return a / b
```

Nothing stops you from calling `safe_divide("x", "y")` — Python never checks
the annotation at runtime. The hint exists for two audiences: a static
checker (`mypy`, `pyright`) run separately, and a human reading the
signature. `Optional[float]` (spelled `float | None` since 3.10) is doing
real work here regardless: it tells the caller **failure is a value they
must check**, not an exception they might forget to catch.

```python
>>> safe_divide(6, 3)
2.0
>>> safe_divide(1, 0)
>>> # None - printed as nothing at the REPL
```

**In `exercises.py`:** `safe_divide`.

In [ ]:
def safe_divide(a: float, b: float) -> Optional[float]:
    """Divide a by b, or return None instead of raising on b == 0.

    The return type is `Optional[float]` - `float | None` - because failure
    is modelled as a value the caller must check, not an exception they must
    remember to catch. This is the same shape as dict.get's default of None.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§1", globals())

## 2. Generics: one definition, many types

```python
T = TypeVar("T")

class Stack(Generic[T]):
    def push(self, item: T) -> None: ...
    def pop(self) -> T: ...
```

`Generic[T]` is what lets `Stack[int]` and `Stack[str]` be *different types*
to a checker, both built from one class definition. At runtime `T` is inert
— `Stack()` behaves identically no matter what you told the checker it
holds. The payoff is entirely static: `stack.push("x")` on something typed
`Stack[int]` is flagged before the code runs, the same way a wrong function
argument would be.

`first_match` shows the same `TypeVar` tying two parameters together:

```python
def first_match(items: Iterable[T], predicate: Callable[[T], bool]) -> Optional[T]:
```

A checker infers `T` from `items` and then requires `predicate` to accept
that same `T` — passing a `list[int]` with a predicate typed for `str`
is an error, without either parameter needing to name the other's type.

```python
>>> stack = Stack()
>>> stack.push(1); stack.push(2)
>>> stack.pop()
2
>>> first_match([1, 3, 4, 6], lambda x: x % 2 == 0)
4
```

**In `exercises.py`:** `Stack.push`, `Stack.pop`, `Stack.peek`,
`Stack.is_empty`, `Stack.__len__`, `first_match`.

In [ ]:
class Stack(Generic[T]):
    """A LIFO stack, generic over the element type.

    `Generic[T]` is what lets a type checker infer `Stack[int]` from
    `Stack()` used with ints, and flag `stack.push("x")` on a `Stack[int]`
    as an error - all at zero runtime cost, since Python itself never
    checks T.
    """

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        raise NotImplementedError("your code here")

    def pop(self) -> T:
        raise NotImplementedError("your code here")

    def peek(self) -> T:
        raise NotImplementedError("your code here")

    def is_empty(self) -> bool:
        raise NotImplementedError("your code here")

    def __len__(self) -> int:
        raise NotImplementedError("your code here")

In [ ]:
def first_match(items: Iterable[T], predicate: Callable[[T], bool]) -> Optional[T]:
    """Return the first item for which predicate(item) is True, else None.

    Generic over T via the TypeVar bound in both the Iterable and the
    Callable - a type checker ties the element type of `items` to the
    argument type `predicate` expects, so passing a `list[int]` with a
    `Callable[[str], bool]` predicate is flagged before you run anything.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§2", globals())

## 3. `@dataclass(frozen=True)`: immutable and hashable

```python
@dataclass(frozen=True)
class Point:
    x: float
    y: float
```

One decorator generates `__init__`, `__repr__`, `__eq__` (by field values),
and — because `frozen=True` — a `__setattr__` that raises, plus `__hash__`.
That last part is the detail worth stating out loud: **a plain
`@dataclass` is unhashable by default the moment you define `__eq__`**
(same rule as hand-written classes — see module 10), but `frozen=True`
tells the generator the instance cannot change, so hashing by field values
is safe and it adds `__hash__` back.

```python
>>> p1, p2 = Point(0, 0), Point(3, 4)
>>> p1.distance_to(p2)
5.0
>>> p1.x = 99
Traceback (most recent call last):
    ...
dataclasses.FrozenInstanceError: cannot assign to field 'x'
>>> {p1, Point(0, 0)}          # hashable - the duplicate collapses
{Point(x=0, y=0)}
```

**In `exercises.py`:** `Point.distance_to`.

In [ ]:
@dataclass(frozen=True)
class Point:
    """An immutable 2D point.

    frozen=True makes __setattr__ raise, which is what lets a Point be
    hashable (dataclass only generates __hash__ when frozen=True or
    eq=False) and safe to use as a dict key or put in a set.
    """

    x: float
    y: float

    def distance_to(self, other: "Point") -> float:
        raise NotImplementedError("your code here")

In [ ]:
lab.check("§3", globals())

## 4. `order=True` and `field(compare=False)`

```python
@dataclass(order=True)
class SemVer:
    major: int
    minor: int
    patch: int
    prerelease: str = field(default="", compare=False)
```

`order=True` generates `__lt__`, `__le__`, `__gt__`, `__ge__` by comparing
fields **in declaration order**, as a tuple. `field(compare=False)` pulls
`prerelease` out of that tuple — and out of `__eq__` too — so two versions
that differ only in prerelease tag compare equal and sort adjacently.
Without it, `order=True` alone would try to compare `prerelease` strings
after the version numbers already decided the ordering — usually harmless,
but here it is explicitly not the intent, so it is excluded.

```python
>>> SemVer(1, 2, 3, "rc1") == SemVer(1, 2, 3, "rc2")
True
>>> sort_versions([SemVer(2, 0, 0), SemVer(1, 5, 0), SemVer(1, 0, 0)])
[SemVer(major=1, minor=0, patch=0, ...), SemVer(major=1, minor=5, patch=0, ...), SemVer(major=2, minor=0, patch=0, ...)]
```

**In `exercises.py`:** `SemVer.__str__`, `sort_versions`.

In [ ]:
@dataclass(order=True)
class SemVer:
    """A semantic version, orderable by (major, minor, patch) only.

    `field(compare=False)` excludes `prerelease` from both __eq__ and the
    ordering methods order=True generates - so 1.2.3-rc1 and 1.2.3 compare
    equal and sort adjacently, which matches how semver defines precedence
    for the release core (real semver also orders prereleases below their
    release, which this simplified version does not model).
    """

    major: int
    minor: int
    patch: int
    prerelease: str = field(default="", compare=False)

    def __str__(self) -> str:
        raise NotImplementedError("your code here")

In [ ]:
def sort_versions(versions: list[SemVer]) -> list[SemVer]:
    """Sort SemVer instances using the dataclass-generated ordering."""
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§4", globals())

## 5. `default_factory`: the mutable-default trap, fixed

```python
@dataclass
class ShoppingCart:
    items: list = field(default_factory=list)
```

`items: list = []` would bind **one list object**, created once when the
class body runs, shared by every instance that does not pass its own —
the same bug as `def f(x, items=[])`. Dataclasses actively forbid this: a
bare mutable literal default raises `ValueError` at class-definition time,
not silently at first use. `field(default_factory=list)` instead stores a
zero-argument callable, invoked fresh for every instance.

```python
>>> a, b = ShoppingCart(), ShoppingCart()
>>> a.add_item("apple")
>>> a.total_items(), b.total_items()
(1, 0)
>>> a.items is b.items
False
```

**In `exercises.py`:** `ShoppingCart.add_item`, `ShoppingCart.total_items`.

In [ ]:
@dataclass
class ShoppingCart:
    """A cart whose item list defaults to empty, per-instance.

    `items: list = []` as a default would share ONE list across every
    instance that doesn't pass its own - the classic mutable-default bug.
    `field(default_factory=list)` calls list() fresh for each instance
    instead, which is why dataclasses forbids a bare mutable literal
    default and raises ValueError at class-definition time if you try.
    """

    items: list = field(default_factory=list)

    def add_item(self, name: str) -> None:
        raise NotImplementedError("your code here")

    def total_items(self) -> int:
        raise NotImplementedError("your code here")

In [ ]:
lab.check("§5", globals())

## 6. `__post_init__`: validation after construction

```python
@dataclass
class PositiveInt:
    value: int

    def __post_init__(self) -> None:
        if self.value <= 0:
            raise ValueError(...)
```

The generated `__init__` only assigns fields — it has no way to express
"and also check this." `__post_init__` runs once, immediately after that
assignment, and is the standard place to put validation a field
declaration alone cannot: cross-field checks, range checks, normalising a
string's case. Raising here means an invalid instance is never
constructed — there is no window where `PositiveInt(-1)` exists half-built.

```python
>>> PositiveInt(5).value
5
>>> PositiveInt(-1)
Traceback (most recent call last):
    ...
ValueError: value must be positive, got -1
```

**In `exercises.py`:** `PositiveInt.__post_init__`.

In [ ]:
@dataclass
class PositiveInt:
    """A dataclass that validates its own field after construction.

    __post_init__ runs once, right after the generated __init__ finishes
    assigning fields - the standard place to add validation a plain
    dataclass field declaration cannot express.
    """

    value: int

    def __post_init__(self) -> None:
        raise NotImplementedError("your code here")

In [ ]:
lab.check("§6", globals())

## 7. `Protocol`: typing by shape, not by ancestry

```python
@runtime_checkable
class SupportsArea(Protocol):
    def area(self) -> float: ...
```

`Circle` and `Rectangle` below share **no base class** — nothing inherits
from `SupportsArea`. It still type-checks to pass either one to
`total_area(shapes: Iterable[SupportsArea])`, because `Protocol` describes
a *shape* (anything with a no-argument `area() -> float`), not a lineage.
This is what "structural typing" means, and it is how Python narrows the
gap with languages that have interfaces natively — you get the type
safety without forcing unrelated classes into an inheritance hierarchy
just so a function can accept both.

`@runtime_checkable` is the detail that makes `isinstance(obj, SupportsArea)`
work at all. Without it, `Protocol` is a static-only construct and
`isinstance` against it raises `TypeError`. Even with it, `isinstance` only
checks that the *names* exist — it does not verify `area` is callable with
the right signature or actually returns a `float`.

```python
>>> total_area([Circle(1), Rectangle(2, 3)])
9.141592653589793
>>> satisfies_supports_area(Circle(1))
True
>>> satisfies_supports_area(object())
False
```

**In `exercises.py`:** `Circle.area`, `Rectangle.area`, `total_area`,
`satisfies_supports_area`.

In [ ]:
@runtime_checkable
class SupportsArea(Protocol):
    """Structural type: anything with a no-arg `area() -> float` method.

    Nothing has to inherit from SupportsArea to satisfy it - Circle and
    Rectangle below are plain classes. @runtime_checkable is what makes
    isinstance(obj, SupportsArea) actually work; without it isinstance
    raises TypeError, since Protocol is a static-only tool by default.
    """

    def area(self) -> float:
        raise NotImplementedError("your code here")

In [ ]:
@dataclass
class Circle:

    radius: float

    def area(self) -> float:
        raise NotImplementedError("your code here")

In [ ]:
@dataclass
class Rectangle:

    width: float
    height: float

    def area(self) -> float:
        raise NotImplementedError("your code here")

In [ ]:
def total_area(shapes: Iterable[SupportsArea]) -> float:
    """Sum area() across any mix of objects that structurally satisfy it.

    Circle and Rectangle share no base class - this function only ever
    calls .area(), so it does not need one. That is what "structural"
    means: the Protocol describes a shape, not a lineage.
    """
    raise NotImplementedError("your code here")

In [ ]:
def satisfies_supports_area(obj: object) -> bool:
    """True if obj structurally satisfies SupportsArea, via isinstance.

    Works only because SupportsArea is decorated @runtime_checkable -
    isinstance checks for the *presence* of an `area` attribute, not that
    it is callable with the right signature or returns a float.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§7", globals())

---

## The exercises

| Function / class | What it teaches |
|---|---|
| `safe_divide` | `Optional[T]` as a failure-modelling return type |
| `Stack`, `first_match` | `Generic[T]` and `TypeVar` — zero runtime cost, real static leverage |
| `Point` | `frozen=True` gives immutability and restores `__hash__` |
| `SemVer`, `sort_versions` | `order=True`; `field(compare=False)` |
| `ShoppingCart` | `field(default_factory=...)` fixes the mutable-default bug |
| `PositiveInt` | `__post_init__` for validation |
| `Circle`, `Rectangle`, `total_area`, `satisfies_supports_area` | `Protocol` and structural typing; `@runtime_checkable` |

Every function in this table has a stub in `exercises.py` and test coverage
in `test_exercises.py` — a clean 1:1 mapping.

## Check your understanding

1. **Does `def f(x: int) -> int` stop you from calling `f("hello")`?**
   <details><summary>Answer</summary>No. Type hints are not enforced at
   runtime by the Python interpreter — they are checked only by a separate
   static tool (mypy, pyright) you run yourself, or ignored entirely.
   Calling <code>f("hello")</code> runs exactly as if the hint were not
   there, unless something inside <code>f</code> happens to fail on a
   string.</details>

2. **Why does `@dataclass(frozen=True)` restore `__hash__`, when a plain
   `@dataclass` with a custom `__eq__` does not have one?**
   <details><summary>Answer</summary>Defining <code>__eq__</code> normally
   disables the default <code>__hash__</code>, because a hash computed
   from mutable fields could go stale the moment a field changes, breaking
   dict/set invariants. <code>frozen=True</code> guarantees the instance
   can never change after construction, so hashing by field values is safe
   again, and the dataclass machinery adds <code>__hash__</code> back.</details>

3. **What does `field(default_factory=list)` fix that `items: list = []`
   would break?**
   <details><summary>Answer</summary><code>items: list = []</code> would
   create one list object when the class body executes, shared by every
   instance that doesn't pass its own — mutating it on one instance would
   affect all of them. <code>default_factory</code> stores a callable that
   dataclasses invokes fresh for each new instance, so every instance gets
   its own list.</details>

4. **What's the difference between `Protocol` and an abstract base class
   (`ABC`) for describing "anything with an `area()` method"?**
   <details><summary>Answer</summary>An ABC requires the concrete classes
   to explicitly inherit from it. A <code>Protocol</code> is structural —
   any class with a matching <code>area()</code> method satisfies it
   automatically, with no inheritance and no change to the concrete
   classes at all. This matters most when you don't control those
   classes, or don't want to force unrelated types into one hierarchy.</details>

5. **Why does `isinstance(obj, SomeProtocol)` raise `TypeError` unless the
   protocol is decorated `@runtime_checkable`?**
   <details><summary>Answer</summary><code>Protocol</code> is designed as
   a static-typing tool by default — it isn't meant to add runtime
   overhead to every class definition. <code>@runtime_checkable</code>
   opts a specific protocol into supporting <code>isinstance</code>,
   which then checks only for the presence of the named
   methods/attributes, not their signatures or return types.</details>

## Interview questions this module answers

- *Do type hints actually enforce anything?*
- *Why would you reach for `@dataclass` instead of writing `__init__` by
  hand?*
- *What's the mutable default argument bug, and how does `dataclass` avoid
  it?*
- *When would you use a `Protocol` instead of an abstract base class?*
- *Why is `frozen=True` the thing that makes a dataclass hashable?*

## Finished? Check everything, then record it

The first cell checks the whole unit. The second records your result in LinguAgent (needs your lab token from [https://linguagent.com/settings#labs](https://linguagent.com/settings#labs) — in Colab, add it once as a secret named `LINGUAGENT_TOKEN` under 🔑 in the left bar).

In [ ]:
lab.check_all(globals())

In [ ]:
lab.sync()

**Next:** [Python 8 - Functional patterns](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/python/17-functional-patterns/lab.ipynb) · [the lesson in LinguAgent](https://linguagent.com/courses/python/typing-and-dataclasses)